# Modified Mutation Signature analysis

In [ ]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

## Somatic data from cBioPortal, unfiltered for Oncogenicity:

In [288]:
start_time=time.asctime(time.localtime())
print(start_time)
#setting path to input and output files
#i) 
#parentpath="INSERT PARENT PATH TO FOLDER WHERE YOU WILL CREATE SUB-FOLDERS PER GENE OF INTEREST AND OUTPUT EXTRACTED DATA"
parentpath="/Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list"

os.chdir(parentpath)
oncotreeannotated_cbio=pd.read_csv("cBioPortal/cBio_all_clinical_fromRStudio_6-14-23_OncoTreeannotated_10-21-24.txt", sep="\t")
cbioallconcat=pd.DataFrame()

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4 for this study) and Gene ID
#ii) USER INPUT FILE NAME 
#TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE SELECT TRANSCRIPT ID", sep="\t")
TranscriptID= pd.read_csv("GL_S_Pipeline_1st_pass_47TSGs_May2023/TSGfolder_MANESelect_names_aalength_GeneIDs_6-13-24.txt", sep="\t") 

# read file with clinical data (R script extraction)
cbio_sampleinfo=pd.read_csv("cBioPortal/cBio_all_clinical_fromRStudio_6-14-23.txt", sep="\t")
# filter down to rows that contain sampleid + Tumor Mutation Burden (TMB) information:
cbio_sampleinfo_tmb=cbio_sampleinfo[cbio_sampleinfo["clinicalAttributeId"].str.contains("TMB")==True]
cbio_sampleinfo_cancertype=cbio_sampleinfo[cbio_sampleinfo["clinicalAttributeId"].str.contains("CANCER_TYPE_DETAILED")==True]
# convert TMB values column to numeric to be able to filter by value:
cbio_sampleinfo_tmb["value"]=pd.to_numeric(cbio_sampleinfo_tmb["value"])
# filter to TMB greater than 10 - make list of samples to be exluded from analysis:
cbio_sampleinfo_tmb_gt10=cbio_sampleinfo_tmb[cbio_sampleinfo_tmb["value"]>10]
Hypermutators=cbio_sampleinfo_tmb_gt10

os.chdir("GL_S_Pipeline_1st_pass_47TSGs_May2023")

# Loop through each folder name aka each gene in the list to read the respective extracted variant file 
for index, row in TranscriptID.iterrows():
    
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #first load MANE Select/transcript identifier to a variable
    transcriptname=TranscriptID.loc[index]["MANE Select Transcript ID"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        #cBio raw data file name starts with 'cBio_' so find that filename from list of filenames in folder and load into pandas df (short for dataframe)
        infile_cbio=f"cBio_{folder_name}_fromRStudio_"
        for file in filelist:
            if file.startswith(infile_cbio):
                cbio_rawdata = pd.read_csv(file, sep="\t")
                
                
        # count of variants:
        One_cbioall=len(cbio_rawdata)
        
        #cbio processing:
        #removing uncalled and germline variants
        cbio_rawdata_tos=cbio_rawdata[cbio_rawdata['mutationStatus'].str.contains("UNCALLED|Germline|GERMLINE|germline")==True]
        cbio_rawdata_s = cbio_rawdata.drop(cbio_rawdata_tos.index)
        
        #s=somatic
        Two_somatic=len(cbio_rawdata_s)
        
        #u=unique meaning removing duplicate variants in patients with multiple samples (by patient id)
        cbio_rawdata_s_u=cbio_rawdata_s.drop_duplicates(subset=['patientId', 'chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Three_uniqueperpatient=len(cbio_rawdata_s_u)
        
        # filter out the variants that have 0 reads of the tumor allele (likely invalid entries)
        cbio_rawdata_s_u_0tumorAltCount=cbio_rawdata_s_u[cbio_rawdata_s_u["tumorAltCount"]==0]
        cbio_rawdata_s_u_no0tumorAltCount=cbio_rawdata_s_u.drop(cbio_rawdata_s_u_0tumorAltCount.index)
        print(len(cbio_rawdata_s_u_no0tumorAltCount))
        Four_no0tumorAltCount=len(cbio_rawdata_s_u_no0tumorAltCount)
        cbio_rawdata_s_u=cbio_rawdata_s_u_no0tumorAltCount
        
        #QC step error checking for genomic coordinates
        #inserting column for QC result
        cbio_rawdata_s_u.insert(1,"QCgenomiccoordinates","NaN")
        #looping over every row in dataframe
        for index, row in cbio_rawdata_s_u.iterrows():
            #assigning row and column indexers to respective variables to be able to call a specific position in df
            rowindex=cbio_rawdata_s_u.index.get_loc(index)
            colindex=cbio_rawdata_s_u.columns.get_loc('QCgenomiccoordinates')
            #length of ref allele minus 1 should match difference between stop and start (except for insertions where diff should be 1 and ref="-")
            if (int(row['endPosition'])-int(row['startPosition']))==(len(row["referenceAllele"])-1):
                cbio_rawdata_s_u.iloc[rowindex,colindex]="Pass"
            else:
                if ((int(row['endPosition'])-int(row['startPosition']))==1)&(row["referenceAllele"]=="-"): 
                    cbio_rawdata_s_u.iloc[rowindex,colindex]="Pass"
                #if criteria not satisfied, variant Fails QC
                else:
                    cbio_rawdata_s_u.iloc[rowindex,colindex]="Fail"
        #filter out failed variants at QC step
        cbio_rawdata_s_u_p=cbio_rawdata_s_u[cbio_rawdata_s_u['QCgenomiccoordinates'].str.contains("Pass")==True]
        ##print QC fail variants to new file to see why they failed (f = fail)
        
        os.chdir(parentpath)
        os.chdir("GL_S_Pipeline_1st_pass_47TSGs_May2023")

        Five_passgenomicQC=len(cbio_rawdata_s_u_p)
        #Five_failgenomicQC=len(cbio_rawdata_s_u_f)

        cbio_otannotate=cbio_rawdata_s_u_p.set_index("uniqueSampleKey").join(oncotreeannotated_cbio.set_index("uniqueSampleKey"), how="left", rsuffix="_cbioclinicalfile")
        print(len(cbio_rawdata_s_u_p))
        cbio_otannotate_1ot=cbio_otannotate[cbio_otannotate["unique_count_parents_in_OncoTree"]==1]
        print(len(cbio_otannotate_1ot))
        #display(cbio_otannotate_1ot)
        cbioallconcat=pd.concat([cbioallconcat,cbio_otannotate_1ot])
    
        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        #os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time=time.asctime(time.localtime())
print(end_time)

Tue Aug 19 22:43:29 2025
746
712
571
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023
880
878
823
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023
1524
1519
1199
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023
1102
1099
914
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023
23889
23772
20534
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_Pipeline_1st_pass_47TSGs_May2023
367
366
318
Current directory: /Users/suhasinilulla/Library/CloudStorage/Box-Box/PhD_BCM/Lulla_Plon_Lab/High_priority_gene_list/GL_S_P

## Combine data for all 40 TSGs and format for COSMIC's SigProfiler Assignment tool

In [290]:
cbioallconcat=cbioallconcat.reset_index()

In [292]:
cbioallconcat_formutsign=cbioallconcat.filter(items=["parent_tissue_code","uniqueSampleKey", "variantType", "chr", "startPosition", "endPosition", "referenceAllele", "variantAllele"])



In [293]:
cbioallconcat_formutsign["Project"]="cbio_all"

In [294]:
cbioallconcat_formutsign["Genome"]="GRCh37"
cbioallconcat_formutsign["Type"]="SOMATIC"

In [295]:
new_order = ['Project', 'parent_tissue_code', 'uniqueSampleKey', 'Genome', "variantType", "chr", "startPosition", "endPosition", "referenceAllele", "variantAllele", 'Type']
cbioallconcat_formutsign = cbioallconcat_formutsign[new_order]

In [300]:
cbioallconcat_formutsign.to_csv("cbio_tryallconcat_includenotOLO_8-19-25.txt", sep="\t", index=None)

In [298]:
cbioallconcat["hugoGeneSymbol"].value_counts()

hugoGeneSymbol
TP53       20534
APC         6319
PTEN        3289
NF1         3086
ATM         3043
RB1         2412
BRCA2       2267
CDKN2A      2129
SMARCA4     2127
SMAD4       2059
PTCH1       1485
CDH1        1457
TSC2        1199
BRCA1       1183
DICER1      1163
STK11       1041
BAP1        1023
MSH6         977
TSC1         914
VHL          823
AXIN2        734
PALB2        676
MSH2         675
MEN1         659
NF2          621
MLH1         591
WT1          571
BARD1        549
CHEK2        536
CYLD         490
CDC73        465
SDHA         427
FLCN         410
SMARCB1      407
CDKN1B       348
FH           325
BMPR1A       319
SUFU         318
CEBPA        197
MAX          190
Name: count, dtype: int64

In [299]:
cbioallconcat["parent_tissue_code"].value_counts()

parent_tissue_code
Bowel (BOWEL)                          13398
Lung (LUNG)                            10818
Breast (BREAST)                         5903
Skin (SKIN)                             5438
Uterus (UTERUS)                         4604
Esophagus/Stomach (STOMACH)             4278
Bladder/Urinary Tract (BLADDER)         3717
CNS/Brain (BRAIN)                       3476
Pancreas (PANCREAS)                     3147
Kidney (KIDNEY)                         1807
Head and Neck (HEAD_NECK)               1509
Prostate (PROSTATE)                     1481
Ovary/Fallopian Tube (OVARY)            1423
Lymphoid (LYMPH)                        1384
Biliary Tract (BILIARY_TRACT)           1220
Liver (LIVER)                           1166
Other (OTHER)                            555
Soft Tissue (SOFT_TISSUE)                459
Cervix (CERVIX)                          433
Ampulla of Vater (AMPULLA_OF_VATER)      387
Myeloid (MYELOID)                        370
Bone (BONE)                         

## Used GUI for COSMIC's SigProfiler Assignment tool to assign mutation signatures contributing to SBS profile for 5 tissue types individually (see Methods, Figure 5) 